CELDA 1 — Instalación de librerías

In [ ]:
!pip install scikit-learn pymongo dnspython --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.0 MB/s eta 0:00:00


CELDA 2 — Librerías

In [ ]:
import pandas as pd
import numpy as np
from pymongo import MongoClient
from datetime import datetime, timezone
from google.colab import userdata

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

CELDA 3 — Conexión a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]

MONGO_URI = userdata.get("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["ernesto_investing_ai"]

coleccion_precios = db["precios_ohlcv"]
coleccion_predicciones = db["predicciones"]
coleccion_metricas = db["metricas_modelos"]

try:
    client.admin.command("ping")
    print("✅ Conexión a MongoDB Atlas exitosa.")
except Exception as e:
    print("❌ Error de conexión:", e)

✅ Conexión a MongoDB Atlas exitosa.


CELDA 4 — Función: leer un ticker desde Mongo y reconstruir el DataFrame


In [ ]:
def leer_precios_de_mongo(ticker):
    """
    Lee el documento de precios_ohlcv guardado por el Notebook 1
    y lo reconstruye como DataFrame con índice de fechas.
    """
    doc = coleccion_precios.find_one({"ticker": ticker})

    if doc is None:
        raise ValueError(f"No hay datos en Mongo para {ticker}. Corre primero el Notebook 1.")

    df = pd.DataFrame({
        "Close": doc["close"],
        "SMA20": doc["sma20"],
        "SMA50": doc["sma50"],
        "EMA12": doc["ema12"],
        "EMA26": doc["ema26"],
        "RSI": doc["rsi"],
        "MACD": doc["macd"],
    }, index=pd.to_datetime(doc["fechas"]))

    df.dropna(inplace=True)
    return df

CELDA 5 — Función: entrenar SVC para un ticker

In [ ]:
def entrenar_svc(ticker):
    """
    Pipeline completo de SVC para un ticker: features, target binario,
    normalización, GridSearchCV y métricas.
    """
    df = leer_precios_de_mongo(ticker)

    # Variable objetivo binaria: 1=BUY si el retorno de mañana es positivo
    df["Retorno"] = df["Close"].pct_change()
    df["Trend"] = np.where(df["Retorno"].shift(-1) > 0, 1, 0)
    df.dropna(inplace=True)

    features = ["SMA20", "SMA50", "EMA12", "EMA26", "RSI", "MACD"]
    X = df[features]
    y = df["Trend"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.20, random_state=42, shuffle=False
    )

    parametros = {
        "kernel": ["linear", "rbf", "poly", "sigmoid"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"]
    }

    grid = GridSearchCV(
        SVC(probability=True),
        parametros,
        cv=5,
        scoring="accuracy"
    )
    grid.fit(X_train, y_train)

    modelo = grid.best_estimator_
    pred = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    matriz = confusion_matrix(y_test, pred)

    senales_historicas = [
        "BUY" if x == 1 else "SELL" for x in modelo.predict(X_scaled)
    ]

    confianza_actual = float(
        max(modelo.predict_proba(X_scaled[-1:].reshape(1, -1))[0])
    )

    return {
        "df": df,
        "modelo": modelo,
        "grid": grid,
        "senales_historicas": senales_historicas,
        "senal_actual": senales_historicas[-1],
        "confianza": round(confianza_actual, 4),
        "accuracy": round(float(accuracy), 4),
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "matriz_confusion": matriz.tolist()
    }

CELDA 6 — Función: guardar resultados en Mongo (esquema del contrato de datos)

In [ ]:
def guardar_resultados_svc(ticker, resultado):
    """
    Guarda en 'predicciones' y 'metricas_modelos' siguiendo
    exactamente el esquema del contrato de datos.
    """
    ahora = datetime.now(timezone.utc)

    documento_prediccion = {
        "ticker": ticker,
        "modelo": "SVC",
        "senal_actual": resultado["senal_actual"],
        "senales_historicas": resultado["senales_historicas"],
        "confianza": resultado["confianza"],
        "fecha_actualizacion": ahora
    }

    documento_metricas = {
        "ticker": ticker,
        "modelo": "SVC",
        "accuracy": resultado["accuracy"],
        "precision": resultado["precision"],
        "recall": resultado["recall"],
        "f1": resultado["f1"],
        "matriz_confusion": resultado["matriz_confusion"],
        "hiperparametros": {
            "kernel": resultado["grid"].best_params_["kernel"],
            "C": resultado["grid"].best_params_["C"],
            "gamma": str(resultado["grid"].best_params_["gamma"])
        },
        "fecha_actualizacion": ahora
    }

    coleccion_predicciones.update_one(
        {"ticker": ticker, "modelo": "SVC"},
        {"$set": documento_prediccion},
        upsert=True
    )

    coleccion_metricas.update_one(
        {"ticker": ticker, "modelo": "SVC"},
        {"$set": documento_metricas},
        upsert=True
    )

CELDA 7 — Ejecución: loop sobre los 5 tickers

In [ ]:
resultados_por_ticker = {}

for ticker in TICKERS:
    print(f"Entrenando SVC para {ticker}...")
    try:
        resultado = entrenar_svc(ticker)
        guardar_resultados_svc(ticker, resultado)
        resultados_por_ticker[ticker] = resultado

        print(f"  Señal actual: {resultado['senal_actual']} "
              f"(confianza {resultado['confianza']})")
        print(f"  Accuracy: {resultado['accuracy']} | F1: {resultado['f1']}")
        print(f"  ✅ Guardado en MongoDB.\n")

    except Exception as e:
        print(f"  ❌ Error con {ticker}: {e}\n")

Entrenando SVC para FSM...
  Señal actual: BUY (confianza 0.624)
  Accuracy: 0.5275 | F1: 0.6055
  ✅ Guardado en MongoDB.

Entrenando SVC para VOLCABC1.LM...
  Señal actual: SELL (confianza 0.5538)
  Accuracy: 0.5281 | F1: 0.0
  ✅ Guardado en MongoDB.

Entrenando SVC para ABX.TO...
  Señal actual: BUY (confianza 0.5673)
  Accuracy: 0.5275 | F1: 0.6261
  ✅ Guardado en MongoDB.

Entrenando SVC para BVN...
  Señal actual: BUY (confianza 0.6304)
  Accuracy: 0.5604 | F1: 0.5122
  ✅ Guardado en MongoDB.

Entrenando SVC para BHP...
  Señal actual: BUY (confianza 0.5418)
  Accuracy: 0.5165 | F1: 0.6508
  ✅ Guardado en MongoDB.



CELDA 8 — Verificación final

In [ ]:
print("Verificación de la colección 'predicciones' (modelo SVC):\n")

for doc in coleccion_predicciones.find({"modelo": "SVC"}, {"_id": 0}):
    print(f"{doc['ticker']}: {doc['senal_actual']} (confianza {doc['confianza']})")

print("\nVerificación de la colección 'metricas_modelos' (modelo SVC):\n")

for doc in coleccion_metricas.find({"modelo": "SVC"}, {"_id": 0}):
    print(f"{doc['ticker']}: accuracy={doc['accuracy']}, f1={doc['f1']}, "
          f"kernel={doc['hiperparametros']['kernel']}")

Verificación de la colección 'predicciones' (modelo SVC):

FSM: BUY (confianza 0.624)
VOLCABC1.LM: SELL (confianza 0.5538)
ABX.TO: BUY (confianza 0.5673)
BVN: BUY (confianza 0.6304)
BHP: BUY (confianza 0.5418)

Verificación de la colección 'metricas_modelos' (modelo SVC):

FSM: accuracy=0.5275, f1=0.6055, kernel=sigmoid
VOLCABC1.LM: accuracy=0.5281, f1=0.0, kernel=rbf
ABX.TO: accuracy=0.5275, f1=0.6261, kernel=sigmoid
BVN: accuracy=0.5604, f1=0.5122, kernel=sigmoid
BHP: accuracy=0.5165, f1=0.6508, kernel=sigmoid
